# 03 Cut to the data window and read
0. Only when needed: raw files unzipped / Numbers converted to .csv.
1. The raw exports (`raw_root`) cut to `[window] months_before` (3 or 6) months before the (first) race up to and including the (last) race → `data_root` (interim), one folder per participant.
2. The interim exports read per participant → `cache/<study_id>.pkl` (+ `.xlsx` to check by hand), used by step 04.
Participants already cut / read are skipped, so you can stop (■) and continue later. A participant cut again gets a fresh interim folder.

In [ ]:
import sys
sys.path.append("../src")

from heatwatch.loading import participants as pp, extract, window

## Settings
`months`: from the settings (`[window] months_before`). `study_ids`: only these (always cut and read again). `brands`: only these devices. `redo`: everyone again.

In [ ]:
months = window.MONTHS   # 3 or 6: change [window] months_before in config/settings.toml
study_ids = []          # e.g. ["A07", "HR136"]
brands = []             # e.g. ["Fitbit", "Polar"]
redo = False

## 0. Prepare raw files (only when needed)
Some exports need a hand first, in the **raw** folder, so the cut finds the files:
- `unzip_folders`: every `.gz` in the folder is extracted (`.tar.gz` / `.tgz` into a subfolder, `.gz` next to it; subfolders not searched, `.zip` not handled)
- `numbers_files`: Apple Numbers file → `;`-separated `.csv` (first table; needs `pip install numbers-parser`)
Empty lists = nothing happens. Step 02 shows which participants have `.gz` or `.numbers` files.

In [ ]:
unzip_folders = []    
numbers_files = []    
for folder in unzip_folders:
    window.unzip(folder)
for numbers_file, csv_file in numbers_files:
    window.numbers_to_csv(numbers_file, csv_file)
    print(f"{csv_file}: written")

## 1. Cut to the window

In [ ]:
info = pp.load()
log = window.cut_participants(info, study_ids=study_ids, brands=brands, redo=redo, months=months)
log

## 2. Read the interim exports

In [ ]:
folders = extract.read_participants(info, study_ids=study_ids, brands=brands, redo=redo)